# Part 1 — What Is a Score? Metrics and the Confusion Matrix by Hand

*Evals from First Principles, Part 1.*

Every eval, no matter how sophisticated, reduces to the same three atoms: a **task**, a **gold label**, and a **metric**. Get any one of those wrong and the number you report is noise dressed up as signal. This notebook builds that machinery by hand: a 2x2 confusion matrix counted from 10 graded outputs, the four metrics (accuracy, precision, recall, F1) derived from its four counts, and a second, imbalanced set that shows why accuracy alone can lie to you.

Pure Python, no imports, fully offline and deterministic — every number below is reproducible on a laptop with no network and no API key.

Companion script: `scoring_basics.py`

In [ ]:
# --- The task ---------------------------------------------------------
# A support-ticket classifier decides "urgent" (1) vs "not urgent" (0).
# We have 10 tickets, each with a human GOLD label and the MODEL's PRED.
# gold[i] and pred[i] describe the same ticket.
GOLD = [1, 1, 1, 0, 0, 1, 0, 0, 1, 0]
PRED = [1, 1, 0, 0, 1, 1, 0, 0, 0, 0]

print(f"{len(GOLD)} tickets. gold[i] and pred[i] describe the same ticket.")
print("GOLD:", GOLD)
print("PRED:", PRED)

## Step 1 — The 2x2 confusion matrix

Every binary prediction against a gold label lands in exactly one of four buckets:

- **True Positive (TP)**: gold says urgent, model says urgent — a hit.
- **False Positive (FP)**: gold says not urgent, model says urgent — a false alarm.
- **False Negative (FN)**: gold says urgent, model says not urgent — a miss.
- **True Negative (TN)**: gold says not urgent, model says not urgent — a correct pass.

Every metric in this notebook is arithmetic on just these four counts. Nothing else about the 10 tickets matters once we have TP, FP, FN, TN.

In [ ]:
def confusion(gold, pred):
    """Return (tp, fp, fn, tn) — the four cells of the 2x2, counted by hand."""
    tp = fp = fn = tn = 0
    for g, p in zip(gold, pred):
        if g == 1 and p == 1:
            tp += 1
        elif g == 0 and p == 1:
            fp += 1
        elif g == 1 and p == 0:
            fn += 1
        else:  # g == 0 and p == 0
            tn += 1
    return tp, fp, fn, tn


tp, fp, fn, tn = confusion(GOLD, PRED)
print(f"confusion:  TP={tp}  FP={fp}  FN={fn}  TN={tn}")
# TP=3  FP=1  FN=2  TN=4 — 3 hits, 1 false alarm, 2 misses, 4 correct passes

## Step 2 — Four metrics, four questions

Each metric answers a different question about the same four counts:

- **Accuracy** — of ALL 10 tickets, how many did we get right? `(TP+TN) / n`.
- **Precision** — of the ones we CALLED urgent, how many really were? `TP / (TP+FP)`.
- **Recall** — of the ones that WERE urgent, how many did we catch? `TP / (TP+FN)`.
- **F1** — the harmonic mean of precision and recall, a single number that penalizes a model for being lopsided in either direction.

Each division guards its 0/0 case: no predictions called "urgent" (or no urgent tickets at all) returns `0.0` instead of raising.

In [ ]:
def accuracy(tp, fp, fn, tn):
    """Fraction of ALL predictions that were correct."""
    total = tp + fp + fn + tn
    return (tp + tn) / total if total else 0.0


def precision(tp, fp):
    """Of the ones we CALLED urgent, how many really were? Guard 0/0 -> 0.0."""
    return tp / (tp + fp) if (tp + fp) else 0.0


def recall(tp, fn):
    """Of the ones that WERE urgent, how many did we catch? Guard 0/0 -> 0.0."""
    return tp / (tp + fn) if (tp + fn) else 0.0


def f1(prec, rec):
    """Harmonic mean of precision and recall. Guard 0+0 -> 0.0."""
    return 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0


acc = accuracy(tp, fp, fn, tn)
prec = precision(tp, fp)
rec = recall(tp, fn)
f = f1(prec, rec)
print(f"accuracy  = (TP+TN)/n         = ({tp}+{tn})/{tp+fp+fn+tn} = {acc:.2f}")
print(f"precision = TP/(TP+FP)        = {tp}/({tp}+{fp}) = {prec:.2f}")
print(f"recall    = TP/(TP+FN)        = {tp}/({tp}+{fn}) = {rec:.2f}")
print(f"F1        = 2PR/(P+R)         = {f:.2f}")
# accuracy=0.70  precision=0.75  recall=0.60  F1=0.67

## Step 3 — Why accuracy lies under class imbalance

The balanced set above has 5 urgent tickets out of 10 — accuracy is a reasonable summary there. Real support queues are not balanced: urgent tickets are rare. Take a queue of 20 tickets where only 2 are truly urgent, and score a lazy baseline model that **always** predicts "not urgent," never looking at the ticket at all.

That baseline will be right on all 18 non-urgent tickets by construction, so accuracy alone will look great. Whether it is actually useful depends entirely on what happens to the 2 tickets that mattered — which accuracy cannot tell you, and recall can.

In [ ]:
imb_gold = [1, 1] + [0] * 18            # 2 urgent, 18 not
imb_pred = [0] * 20                      # always "not urgent"

itp, ifp, ifn, itn = confusion(imb_gold, imb_pred)
iacc = accuracy(itp, ifp, ifn, itn)
iprec = precision(itp, ifp)
irec = recall(itp, ifn)
if1 = f1(iprec, irec)

print(f"confusion:  TP={itp}  FP={ifp}  FN={ifn}  TN={itn}")
print(f"accuracy  = (TP+TN)/n         = ({itp}+{itn})/{itp+ifp+ifn+itn} = {iacc:.2f}")
print(f"precision = TP/(TP+FP)        = {itp}/({itp}+{ifp}) = {iprec:.2f}")
print(f"recall    = TP/(TP+FN)        = {itp}/({itp}+{ifn}) = {irec:.2f}")
print(f"F1        = 2PR/(P+R)         = {if1:.2f}")
# accuracy=0.90  precision=0.00  recall=0.00  F1=0.00
print("\n-> 90% accurate, 0% recall. Accuracy hid a model that is useless")
print("   at the only job that matters. This is why we need the whole 2x2.")

## Recap

- A score is only ever as trustworthy as its three atoms: the **task** definition, the **gold** label, and the **metric**. Change any one and the number changes meaning.
- The **2x2 confusion matrix** (TP, FP, FN, TN) is the primitive underneath every classification metric — accuracy, precision, recall, and F1 are all just arithmetic on those four counts, nothing more.
- **Precision** and **recall** ask different questions (what we called right vs. what we caught), and **F1** collapses them into one number without letting either hide.
- **Accuracy alone lies under class imbalance**: the always-negative baseline scored 90% accuracy while catching zero of the urgent tickets it existed to catch. Whenever the positive class is rare, look at the whole confusion matrix, not just accuracy.

Next: golden sets — where the GOLD labels above actually come from, and what makes one trustworthy.